# Using Predictive Analytics to Identify Engagement Drivers in Instagram Reels
## Notebook 1 of 4: Business Understanding and Data Understanding
**CRISP-DM phases 1 and 2**

### Project roadmap (CRISP-DM)

| CRISP-DM phase | Notebook | What happens there |
|---|---|---|
| 1. Business Understanding | **`01_business_and_data_understanding`** | Objective, analytics goal, success criteria, risks |
| 2. Data Understanding | **`01_business_and_data_understanding`**, then `02_data_preparation` (section 3.4) | Describe the raw data and verify its quality here; explore the cleaned data in depth in Notebook 2 |
| 3. Data Preparation | `02_data_preparation` | Select, clean, construct, and format the data (and explore the cleaned data) |
| 4. Modeling | `03_modeling` | Build, tune, and assess the models |
| 5. Evaluation | `04_evaluation_and_deployment` | Judge results against the business goal and review the process |
| 6. Deployment | `04_evaluation_and_deployment` | Deployment, monitoring, and final review |

Data Understanding happens in two passes. This notebook describes the raw data and records its quality problems without changing anything. Once the data has been cleaned in Notebook 2, it is explored in depth there (section 3.4), because charts and correlations need clean numbers, dates, and labels.

CRISP-DM is iterative rather than strictly linear. In this project, lessons from Evaluation led back to Data Understanding (interactions between features) and Data Preparation (interaction features), followed by a second round of Modeling.

---
# Phase 1: Business Understanding

## 1.1 Background and business objective
Audience engagement with short-form Instagram Reels varies widely across videos, even within the same creator's account. Because creators differ in audience size and baseline performance, raw engagement metrics alone do not provide a consistent basis for comparison.

**Business objective:** identify which content characteristics (how a Reel is framed, structured, and delivered) are associated with an above-baseline result in DIY content, so that creators and social media teams can plan content around repeatable patterns instead of guesswork, independent of creator popularity.

## 1.2 Analytics goal
- **Target variable:** `performance_bucket`, a creator-relative label (high-performing vs. baseline). Defining performance relative to each creator's typical output reduces the bias from differences in follower count and reach.
- **Supporting metric:** `engagement_rate` = (likes + comments) / views, used to give context to the performance groups.
- **Problem type:** binary classification, plus descriptive analysis of which features are associated with high performance.
- **Models:** logistic regression, k-nearest neighbors (k-NN), and a classification tree (Notebook 3).

## 1.3 Success criteria
- **Business:** surface content patterns (hooks, structures, presentation choices) that a team could test in future Reels.
- **Technical:** the models should identify high-performing Reels better than random guessing, tested on Reels they were not trained on (a held-out validation set). Because only about 30% of Reels are high-performing, accuracy is misleading: a model that always predicts “baseline” would be about 70% accurate while finding no high-performing Reels (confirmed in section 2.4). Models are therefore judged on two measures for the high-performing class:
    - **Recall:** of all the Reels that really are high-performing, the share the model finds.
    - **F1:** one score that balances recall with precision (of the Reels the model flags as high-performing, the share that really are).

## 1.4 Assumptions, constraints, and risks
- **Small sample:** The number of Reels is limited, so some categories have very few examples and validation scores will be noisy.
- **Reel age is a confounder:** Metrics were recorded on `collection_date`, and the high-performing Reels turn out to be much older than the baseline Reels on that date (Notebook 2, section 3.4). Age is tied to the outcome and may also be tied to some content features, so content effects cannot be cleanly separated from age. How the high and baseline Reels were sampled should be confirmed.
- **Missing drivers:** Timing, audience behavior, and algorithmic distribution are not in the dataset and may explain a large share of performance.

## 1.5 Project plan
The roadmap above maps each phase to a notebook. The data file used here is `../Data/raw_data.csv` (one row per Reel).

---
# Phase 2: Data Understanding

## 2.1 Collect initial data

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)

In [ ]:
reels_df = pd.read_csv('../Data/raw_data.csv')

print(f"Rows: {reels_df.shape[0]}  |  Columns: {reels_df.shape[1]}")
reels_df.head(10)

## 2.2 Describe the data

### Data dictionary (raw file)

| Column | Description | Variable type | Raw format |
|---|---|---|---|
| `creator_handle` | Account name of the creator | Categorical (identifier) | text |
| `total_followers` | Followers of the account | Numerical | text such as `953K` |
| `url` | Link to the Reel (contains the unique Reel ID) | Identifier | text |
| `date_posted` | Date the Reel was published | Date | text `m/d/yy` |
| `collection_date` | Date performance was recorded | Date | text `m/d/yy` |
| `performance_bucket` | Creator-relative performance: high or baseline | Categorical (**target**) | `baseline` / `high` |
| `views_raw`, `likes_raw`, `comments_raw` | Views, likes, comments | Numerical | text such as `113K` |
| `engagement_rate` | Likes and comments relative to views | Numerical | empty (recalculated in Data Preparation) |
| `reel_structure` | How the Reel is organized (list, process, transformation, narrative, showcase) | Categorical | text |
| `craft_category` | Type of craft shown | Categorical | text |
| `hook_text` | Text of the opening hook | Free text | text, missing when the hook is visual |
| `hook_category` | Framing used to attract attention | Categorical | text |
| `pov_present` | Point-of-view framing present | Binary | `Y` / `N` |
| `result_shown_first` | Final result shown in the opening frames | Binary | `Y` / `N` |
| `voice_present` | Spoken audio present | Binary | `Y` / `N` |
| `audio_not_required` | Message understandable without audio | Binary | `Y` / `N` |

Although `performance_bucket` and the delivery variables have only two values, they are treated as categorical (class membership or presence/absence), and no numeric relationship between the values is assumed.

In [ ]:
reels_df.info()

## 2.3 Verify data quality

In [ ]:
print("Missing values per column:")
print(reels_df.isna().sum(), "\n")

print("Duplicate rows:", reels_df.duplicated().sum())
print("Duplicate URLs:", reels_df['url'].duplicated().sum())

Although no rows are identical, some URLs appear more than once. Checking which columns differ between the copies:

In [ ]:
dup_rows = reels_df[reels_df['url'].duplicated(keep=False)].sort_values('url')

# Which columns differ between the copies of the same URL?
differing = {}
for url, group in dup_rows.groupby('url'):
    for col in reels_df.columns:
        if col != 'url' and group[col].astype(str).nunique() > 1:
            differing[col] = differing.get(col, 0) + 1

print(f"{len(dup_rows)} rows share {dup_rows['url'].nunique()} URLs")
print("Columns that differ between copies (number of URLs affected):", differing)
dup_rows[['url', 'creator_handle', 'date_posted', 'performance_bucket', 'views_raw', 'craft_category']].head(10)

In [ ]:
binary_cols = ['pov_present', 'result_shown_first', 'voice_present', 'audio_not_required']

# Raw formats that will need standardizing in Data Preparation
print(reels_df[['total_followers', 'views_raw', 'likes_raw', 'comments_raw']].head(8), "\n")
print("date_posted examples:      ", reels_df['date_posted'].astype(str).head(5).tolist())
print("collection_date examples:  ", reels_df['collection_date'].astype(str).head(5).tolist(), "\n")

for col in ['craft_category', 'reel_structure', 'hook_category']:
    print(f"--- {col} ---")
    print(reels_df[col].value_counts(dropna=False), "\n")

for col in binary_cols:
    print(col, reels_df[col].astype(str).str.strip().str.upper().value_counts(dropna=False).to_dict())

### Data quality findings and where each is handled
This notebook only records the problems. All cleaning is done once, in Notebook 2.

| Finding | Handled in Notebook 2 |
|---|---|
| `engagement_rate` is empty for every row | Recalculated from likes, comments, and views (section 3.3) |
| `hook_text` is missing for some Reels because the hook is conveyed visually | Not imputed; the column is dropped and not used for modeling (section 3.1) |
| Some Reels (the same URL) appear **twice with identical content but opposite performance labels** | Resolved with the `DUPLICATE_RULE` setting (section 3.1). Left as they are, they would add label noise, and a copy could land in both the training and validation sets and inflate scores |
| URLs use several formats (`/reel/<id>/`, `/reels/<id>/`, `/p/<id>/`), and one is a profile link with no Reel ID | The Reel ID is extracted with a pattern that covers all formats; the full URL is kept when there is no ID (section 3.1) |
| Follower, view, like, and comment counts are text with `K`/`M` suffixes (for example `953K`) | Converted to numbers (section 3.2) |
| Dates are text in `m/d/yy` format | Parsed to dates (section 3.2) |
| Category labels vary in case and spacing, and the `Y`/`N` columns include a few lowercase `y` values | Standardized and encoded (section 3.2) |

## 2.4 Look at the target variable

In [ ]:
target_counts = reels_df['performance_bucket'].str.strip().str.lower().value_counts()
high_share = target_counts['high'] / target_counts.sum()

print(target_counts)
print(f"\nShare of high-performing reels: {high_share:.1%}")
print(f"Accuracy of a model that always predicts 'baseline': {1 - high_share:.1%}")

Only about 30% of Reels are high-performing, so the classes are imbalanced. Because of this, models are judged on recall and F1 for the high-performing class instead of accuracy (section 1.3), and the train/validation split is stratified, meaning both sets keep the same share of high-performing Reels.

## 2.5 Summary and next steps

| Finding | Next Steps |
|---|---|
| About 30% of Reels are high-performing | Use a stratified split and judge models on F1 and recall for the high-performing class (Notebook 3) |
| The raw data has formatting problems and conflicting duplicate labels (section 2.3) | Clean it in Notebook 2, sections 3.1 and 3.2 |
| Distributions, relationships between variables, interactions, and the effect of Reel age need clean numbers, dates, and labels to explore | Explore the cleaned data in Notebook 2, section 3.4 (the second pass of Data Understanding) |